# Report Tables Sanity Check

This notebook recreates the 4 report tables directly from finalized run outputs for visual verification.

Hardcoded inputs:
- Run A (MiniLM): `results/grid_runs/20260328_191443`
- Run B (MPNet): `results/grid_runs/20260406_235803`
- Comparison output: `results/comparisons/20260328_191443_vs_20260406_235803/overall_by_method.csv`

In [5]:
ROOT, ROOT / "results" / "grid_runs" / "20260328_191443", ROOT / "results/grid_runs/20260328_191443"

(WindowsPath('C:/Users/User/Downloads/COMP5801/Proposal/project'),
 WindowsPath('C:/Users/User/Downloads/COMP5801/Proposal/project/results/grid_runs/20260328_191443'),
 WindowsPath('C:/Users/User/Downloads/COMP5801/Proposal/project/results/grid_runs/20260328_191443'))

In [6]:
from pathlib import Path
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().parent
RUN_A = ROOT / "results" / "grid_runs" / "20260328_191443"
RUN_B = ROOT / "results" / "grid_runs" / "20260406_235803"
COMPARE = ROOT / "results" / "comparisons" / "20260328_191443_vs_20260406_235803" / "overall_by_method.csv"

A_SUMMARY = RUN_A / "aggregate" / "summary.csv"
B_SUMMARY = RUN_B / "aggregate" / "summary.csv"

print("A summary:", A_SUMMARY)
print("B summary:", B_SUMMARY)
print("Compare CSV:", COMPARE)

a = pd.read_csv(A_SUMMARY)
b = pd.read_csv(B_SUMMARY)
cmp = pd.read_csv(COMPARE)

for name, df in [("A", a), ("B", b), ("CMP", cmp)]:
    print(f"{name} rows={len(df)}")

A summary: C:\Users\User\Downloads\COMP5801\Proposal\project\results\grid_runs\20260328_191443\aggregate\summary.csv
B summary: C:\Users\User\Downloads\COMP5801\Proposal\project\results\grid_runs\20260406_235803\aggregate\summary.csv
Compare CSV: C:\Users\User\Downloads\COMP5801\Proposal\project\results\comparisons\20260328_191443_vs_20260406_235803\overall_by_method.csv
A rows=60
B rows=60
CMP rows=3


In [7]:
def _table_original(df, dataset):
    cols = ["method", "MRR", "Recall@1", "Recall@5", "Recall@10", "NDCG@10", "time_seconds"]
    out = df[(df["dataset"] == dataset) & (df["chunk_size"] == "original")][cols].copy()
    order_method = {"dense": 0, "sparse": 1, "hybrid": 2}
    out["_m"] = out["method"].map(order_method)
    out = out.sort_values(["_m"]).drop(columns=["_m"]).reset_index(drop=True)
    out = out.rename(columns={"method": "Method", "Recall@1": "R@1", "Recall@5": "R@5", "Recall@10": "R@10", "time_seconds": "Time (s)"})
    out["Method"] = out["Method"].str.capitalize()
    out[["MRR", "R@1", "R@5", "R@10", "NDCG@10", "Time (s)"]] = out[["MRR", "R@1", "R@5", "R@10", "NDCG@10", "Time (s)"]].round(3)
    return out[["Method", "MRR", "R@1", "R@5", "R@10", "NDCG@10", "Time (s)"]]

def _table_chunk_nfcorpus(df):
    cols = ["method", "chunk_size", "MRR", "Recall@5", "Recall@10", "NDCG@10"]
    out = df[df["dataset"] == "nfcorpus"][cols].copy()
    order_chunk = {"original": 0, "chunk_128": 1, "chunk_256": 2, "chunk_512": 3}
    order_method = {"dense": 0, "sparse": 1, "hybrid": 2}
    out["_m"] = out["method"].map(order_method)
    out["_c"] = out["chunk_size"].map(order_chunk)
    out = out.sort_values(["_m", "_c"]).drop(columns=["_m", "_c"]).reset_index(drop=True)
    out = out.rename(columns={"method": "Method", "chunk_size": "Chunk", "Recall@5": "R@5", "Recall@10": "R@10"})
    out["Method"] = out["Method"].str.capitalize()
    out["Chunk"] = out["Chunk"].replace({"chunk_128": "128 tok", "chunk_256": "256 tok", "chunk_512": "512 tok"})
    out[["MRR", "R@5", "R@10", "NDCG@10"]] = out[["MRR", "R@5", "R@10", "NDCG@10"]].round(3)
    return out[["Method", "Chunk", "MRR", "R@5", "R@10", "NDCG@10"]]

table1_nfcorpus = _table_original(a, "nfcorpus")
table2_scifact = _table_original(a, "scifact")
table3_chunk_nf = _table_chunk_nfcorpus(a)

print("Table 1 (nfcorpus, original docs)")
display(table1_nfcorpus)
print("Table 2 (scifact, original docs)")
display(table2_scifact)
print("Table 3 (nfcorpus chunk-size effect)")
display(table3_chunk_nf)

Table 1 (nfcorpus, original docs)


,Method,MRR,R@1,R@5,R@10,NDCG@10,Time (s)
0,Dense,0.544,0.051,0.124,0.160,0.345,22.906
1,Sparse,0.559,0.069,0.134,0.167,0.349,0.439
2,Hybrid,0.591,0.066,0.144,0.171,0.373,24.232


Table 2 (scifact, original docs)


,Method,MRR,R@1,R@5,R@10,NDCG@10,Time (s)
0,Dense,0.606,0.476,0.731,0.772,0.637,32.540
1,Sparse,0.652,0.532,0.751,0.798,0.677,2.071
2,Hybrid,0.676,0.544,0.777,0.828,0.703,35.239


Table 3 (nfcorpus chunk-size effect)


,Method,Chunk,MRR,R@5,R@10,NDCG@10
0,Dense,original,0.544,0.124,0.160,0.345
1,Dense,128 tok,0.539,0.083,0.116,0.380
2,Dense,256 tok,0.533,0.098,0.138,0.349
3,Dense,512 tok,0.540,0.119,0.166,0.343
4,Sparse,original,0.559,0.134,0.167,0.349
5,Sparse,128 tok,0.545,0.097,0.115,0.366
6,Sparse,256 tok,0.553,0.122,0.145,0.356
7,Sparse,512 tok,0.569,0.138,0.167,0.354
8,Hybrid,original,0.591,0.144,0.171,0.373
9,Hybrid,128 tok,0.576,0.103,0.130,0.405


In [8]:
table4_embed_compare = cmp[[
    "method", "embed1_MRR", "embed2_MRR", "embed1_Recall@10", "embed2_Recall@10", "embed1_NDCG@10", "embed2_NDCG@10"
]].copy()

table4_embed_compare = table4_embed_compare.rename(columns={
    "method": "Method",
    "embed1_MRR": "MiniLM MRR",
    "embed2_MRR": "MPNet MRR",
    "embed1_Recall@10": "MiniLM R@10",
    "embed2_Recall@10": "MPNet R@10",
    "embed1_NDCG@10": "MiniLM NDCG@10",
    "embed2_NDCG@10": "MPNet NDCG@10"
})

order_method = {"dense": 0, "sparse": 1, "hybrid": 2}
table4_embed_compare["_m"] = table4_embed_compare["Method"].str.lower().map(order_method)
table4_embed_compare = table4_embed_compare.sort_values(["_m"]).drop(columns=["_m"]).reset_index(drop=True)

table4_embed_compare["Method"] = table4_embed_compare["Method"].str.capitalize()
table4_embed_compare[["MiniLM MRR", "MPNet MRR", "MiniLM R@10", "MPNet R@10", "MiniLM NDCG@10", "MPNet NDCG@10"]] = table4_embed_compare[["MiniLM MRR", "MPNet MRR", "MiniLM R@10", "MPNet R@10", "MiniLM NDCG@10", "MPNet NDCG@10"]].round(5)

print("Table 4 (MiniLM vs MPNet, all datasets/chunks aggregated by method)")
display(table4_embed_compare)

Table 4 (MiniLM vs MPNet, all datasets/chunks aggregated by method)


,Method,MiniLM MRR,MPNet MRR,MiniLM R@10,MPNet R@10,MiniLM NDCG@10,MPNet NDCG@10
0,Dense,0.51067,0.51059,0.39367,0.39367,0.37370,0.37369
1,Sparse,0.48840,0.48836,0.33963,0.33963,0.32335,0.32334
2,Hybrid,0.53337,0.53347,0.38356,0.38355,0.37583,0.37585
